In [ ]:
#linearize fasta
awk '/^>/ {printf("\n%s\n",$0);next; } { printf("%s",$0);}  END {printf("\n");}' < ASV.fna > ASV_lin.fa

In [ ]:
#assign taxonomy with rdp classifer
conda activate rdp_classifer
#wget https://github.com/terrimporter/CO1Classifier/releases/download/v4/CO1v4_trained.tar.gz #training dataset
rdp_classifier -Xmx8g classify -t /Applications/mydata_trained/rRNAClassifier.properties -o ~/Desktop/rdp_out ~/Desktop/MiSeq/COI/ASV_lin.fa#filter output (only keep metazoa, change <70 bootstrap suppport to NA) => 387 metazoan ASVs with phylum assignment remain


In [ ]:
#extract good ASVs
seqtk subseq ASV_lin.fa filtered_ASVs.txt > filtered_ASVs.fa

In [ ]:
#repeat pseudogene filter
java -jar macse_v2.06.jar -prog enrichAlignment -align /Users/susanneKramer/Desktop/COI_database/Midori_alignment/Midori_invert.fa -gc_def 5 -maxSTOP_inSeq 0 -output_only_added_seq_ON =TRUE -fixed_alignment_ON =TRUE -maxDEL_inSeq 5 -maxFS_inSeq 0 -maxINS_inSeq 0 -out_AA /Users/susannekramer/Desktop/RDP_invert_align.faa -seq /Users/susannekramer/Desktop/MiSeq/COI/RDP/filtered_ASVs.fa
java -jar macse_v2.06.jar -prog enrichAlignment -align /Users/susanneKramer/Desktop/COI_database/Midori_alignment/Midori_chordata.fa -gc_def 2 -maxSTOP_inSeq 0 -output_only_added_seq_ON =TRUE -fixed_alignment_ON =TRUE -maxDEL_inSeq 5 -maxFS_inSeq 0 -maxINS_inSeq 0 -out_AA /Users/susannekramer/Desktop/RDP_vert_align.faa -seq /Users/susannekramer/Desktop/MiSeq/COI/RDP/filtered_ASVs.fa
#11 pseudogenes identified
./faSomeRecords -exclude ~/Desktop/MiSeq/COI/RDP/filtered_ASVs.fa ~/Desktop/MiSeq/COI/pseudogenes_RDP.txt ~/Desktop/MiSeq/COI/RDP/RDP_ASVs_clean.fna
seqtk subseq ASV_lin_done.fna RDP/RDP_filtered_ASVs.txt > RDP/RDP_filtered_ASVs.fna

In [ ]:
#Use COIs from dada2 and assign taxonomy using Blast+ and MIDORI2
blastn -db MIDORI2_UNIQ_NUC_SP_GB251_CO1_Blast -query ASV.fna -word_size 11 -max_target_seqs 1 -outfmt 6 -evalue 1e04 -out MIDORI2_uniq_results.txt

In [ ]:
#filter output to hits with id >0.95
awk -F "\t" '{if($4>95) { print }}' MIDORI2_uniq_results.txt > MIDORI2_uniq_results_filtered.txt

In [ ]:
#Extract good ASVs
seqtk subseq ASV.fna MIDORI2/filtered_ASVs.txt > MIDORI2/filtered_ASVs.fna

In [ ]:
#Check for pseudogenes
java -jar macse_v2.06.jar -prog enrichAlignment -align /Users/susanneKramer/Desktop/COI_database/Midori_alignment/Midori_invert.fa -gc_def 5 -maxSTOP_inSeq 0 -output_only_added_seq_ON =TRUE -fixed_alignment_ON =TRUE -maxDEL_inSeq 5 -maxFS_inSeq 0 -maxINS_inSeq 0 -out_AA /Users/susannekramer/Desktop/invert_alignA.faa -seq /Users/susannekramer/Desktop/ASV_A.fasta

In [ ]:
java -jar macse_v2.06.jar -prog enrichAlignment -align /Users/susanneKramer/Desktop/COI_database/Midori_alignment/Midori_chordata.fa -gc_def 2 -maxSTOP_inSeq 0 -output_only_added_seq_ON =TRUE -fixed_alignment_ON =TRUE -maxDEL_inSeq 5 -maxFS_inSeq 0 -maxINS_inSeq 0 -out_AA /Users/susannekramer/Desktop/vert_align.faa -seq /Users/susannekramer/Desktop/ASV_A.fasta

#filter pseudogenes: not included in vert and invert alignment: 6 pseudogenes detected
#remove pseudogenes from filtered_ASVs
./faSomeRecords -exclude ~/Desktop/MiSeq/COI/MIDORI2/filtered_ASVs.fna ~/Desktop/MiSeq/COI/MIDORI2/pseudogenes.txt ~/Desktop/MiSeq/COI/MIDORI2/MIDORI2_ASVs_clean.fna#441 ASVs remaining!


In [ ]:
#use tax_subset.R to remove unassigned and pseudogenes etc. from respective taxonomy files
